In [ ]:
import tushare as ts
import pandas as pd
from config.config import *


In [13]:
# offset后面点，挑点早起成立的基金
bdf = pro.fund_basic(market='O', status='L', offset = '10000',fields='ts_code,name,found_date,fund_type,invest_type,benchmark')
bdf

,ts_code,name,found_date,fund_type,invest_type,benchmark
0,016600.OF,万家品质生活C,20220915,混合型,灵活配置型,沪深300指数收益率*50%+上证国债指数收益率*50%
1,016620.OF,万家颐和C,20220915,混合型,灵活配置型,沪深300指数收益率*50%+上证国债指数收益率*50%
2,016533.OF,嘉实纳斯达克100ETF联接C,20220915,股票型,被动指数型,经估值汇率调整后的纳斯达克100指数收益率*95%+银行活期存款利率(税后)*5%
3,016598.OF,万家鑫安纯债E,20220915,债券型,债券型,中债综合指数(总财富)收益率*90%+1年期定期存款利率(税后)*10%
4,016405.OF,大成景泽中短债C,20220915,债券型,债券型,中债综合财富(1-3年)指数收益率*80%+一年期定期存款利率(税后)*20%
...,...,...,...,...,...,...
13898,100016.OF,富国天源沪港深A,20020816,混合型,平衡型,沪深300指数收益率*65%+中债综合全价指数收益率*30%+同业存款利率*5%
13899,020001.OF,国泰金鹰增长,20020508,混合型,灵活配置型,沪深300指数收益率*80%+中证综合债指数收益率*20%
13900,000001.OF,华夏成长,20011218,混合型,成长型,None
13901,202001.OF,南方稳健成长,20010928,混合型,成长型,None


In [15]:
# from datetime import datetime, timedelta
# # 清洗
# def is_index_fund(row: pd.Series) -> bool:
#     keywords = ['指数', 'ETF', '增强']
#     invest_type = str(row.get('invest_type', ''))
#     name = str(row.get('name', '')).upper()
#     for keyword in keywords:
#         if keyword in invest_type or keyword in name:
#             return True
#     return False

# def list_index_funds(df: pd.DataFrame) -> pd.DataFrame:
#     """筛出指数基金"""
#     df['is_index_fund'] = df.apply(is_index_fund, axis=1)
#     non_index_funds = df[~df['is_index_fund']]
#     return non_index_funds.drop(columns=['is_index_fund'])

# def is_ten_year_old(row: pd.Series) -> bool:
#     """
#     筛选出成立时间 >= 5年 的数据
#     """
#     found_date = str(row.get('found_date', ''))
#     if len(found_date) != 8:
#         return False
#     year = int(found_date[:4])
#     month = int(found_date[4:6])
#     day = int(found_date[6:8])
    
#     found_datetime = datetime(year, month, day)
#     # 5年 = 365 * 5 天（不严格按闰年，和你原逻辑保持一致）
#     five_year_ago = datetime.now() - timedelta(days=365 * 5)
#     return found_datetime <= five_year_ago

# def filter_five_year_old_funds(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     5年为期限，筛出成立满5年的基金
#     返回：成立满5年的基金, 成立不满5年的基金
#     """
#     df['is_ten_year_old'] = df.apply(is_ten_year_old, axis=1)
#     old_funds = df[df['is_ten_year_old']]
#     young_funds = df[~df['is_ten_year_old']]
#     return old_funds.drop(columns=['is_ten_year_old']), young_funds.drop(columns=['is_ten_year_old'])
# def normalize_name(fund_list: pd.DataFrame) -> pd.DataFrame:
#     """筛出AC的"""
#     fund_list = fund_list[~(
#         fund_list['name'].str[:-1].isin(
#             fund_list[fund_list['name'].str.endswith('A')]['name'].str[:-1]
#         ) & fund_list['name'].str.endswith('C')
#     )]
#     return fund_list


In [23]:
from datetime import datetime, timedelta
# 清洗
def is_index_fund(row: pd.Series) -> bool:
    keywords = ['指数', 'ETF', '增强', '债券', '灵活配置', '货币', 'QDII', 'FOF', '平衡', '成长', '香港', '全球', '人民币', '亚洲']
    keywords_inv = ['混合']
    invest_type = str(row.get('invest_type', ''))
    name = str(row.get('name', '')).upper()
    fund_type = str(row.get('fund_type', '')).upper()
    for keyword in keywords:
        if keyword in invest_type or keyword in name or keyword in fund_type:
            return True
    for keyword in keywords_inv:
        if keyword in invest_type:
            return True
    return False

def list_index_funds(df: pd.DataFrame) -> pd.DataFrame:
    """筛出指数基金"""
    df['is_index_fund'] = df.apply(is_index_fund, axis=1)
    non_index_funds = df[~df['is_index_fund']]
    return non_index_funds.drop(columns=['is_index_fund'])

def is_ten_year_old(row: pd.Series) -> bool:
    """
    筛选出成立时间 >= 10年 的数据
    """
    found_date = str(row.get('found_date', ''))
    if len(found_date) != 8:
        return False
    year = int(found_date[:4])
    month = int(found_date[4:6])
    day = int(found_date[6:8])
    
    ten_year_ago = datetime.strptime( "2016-03-30", "%Y-%m-%d")
    found_datetime = datetime(year, month, day)
    return found_datetime <= ten_year_ago

def filter_ten_year_old_funds(df: pd.DataFrame) -> pd.DataFrame:
    """
    10年为期限，筛出成立满10年的基金
    返回：成立满10年的基金, 成立不满10年的基金
    """
    df['is_ten_year_old'] = df.apply(is_ten_year_old, axis=1)
    old_funds = df[df['is_ten_year_old']]
    young_funds = df[~df['is_ten_year_old']]
    return old_funds.drop(columns=['is_ten_year_old']), young_funds.drop(columns=['is_ten_year_old'])
def normalize_name(fund_list: pd.DataFrame) -> pd.DataFrame:
    """筛出AC的"""
    fund_list = fund_list[~(
        fund_list['name'].str[:-1].isin(
            fund_list[fund_list['name'].str.endswith('A')]['name'].str[:-1]
        ) & fund_list['name'].str.endswith('C')
    )]
    return fund_list


In [24]:
b_df = list_index_funds(bdf)
odf, ydf = filter_ten_year_old_funds(b_df)
odf = normalize_name(odf)
odf.to_csv('data/old_funds.csv', index=False, encoding='utf-8-sig')
odf

,ts_code,name,found_date,fund_type,invest_type,benchmark
11232,001917.OF,招商量化精选A,20160315,股票型,股票型,中证500指数收益率*80%+中债综合指数收益率*20%
11233,002229.OF,华夏经济转型,20160315,股票型,股票型,沪深300指数收益率*90%+上证国债指数收益率*10%
11234,001975.OF,景顺长城环保优势,20160315,股票型,股票型,中证环保产业指数收益率*40%+沪深300指数收益率*40%+中证全债指数收益率*20%
11240,002334.OF,汇丰晋信大盘波动精选A,20160311,股票型,股票型,沪深300指数*90%+同业存款利率(税后)*10%
11265,001718.OF,工银物流产业A,20160301,股票型,股票型,沪深300运输指数收益率*80%+中债综合财富(总值)指数收益率*20%
...,...,...,...,...,...,...
13872,160603.OF,鹏华普天收益,20030712,混合型,收益型,沪深300指数收益率*70%+中证综合债指数收益率*30%
13874,070002.OF,嘉实增长,20030709,混合型,增长型,巨潮500(小盘)指数收益率*60%+中债总指数收益率*40%
13876,070003.OF,嘉实稳健,20030709,混合型,稳健型,巨潮200(大盘)指数收益率*60%+中债总指数收益率*40%
13888,090001.OF,大成价值增长A,20021111,混合型,价值型,沪深300指数*80%+中债综合指数*20%


In [24]:
print(datetime.now() - timedelta(days=365 * 10))

2016-04-01 22:04:37.435982


In [ ]:
# import akshare as ak
# import pandas as pd
# from datetime import datetime, timedelta

# # 1. 获取基金基本信息（使用最新的 fund_name_em 接口）
# # 输出列：基金代码, 拼音缩写, 基金简称, 基金类型, 拼音全称
# print("正在获取全量基金列表，请稍候...")
# fund_basic = ak.fund_name_em()   # [citation:4]

# # 筛选主动股票型和混合型基金
# stock_mix_funds = fund_basic[fund_basic['基金类型'].isin(['股票型', '混合型'])].copy()
# print(f"筛选出股票/混合型基金: {len(stock_mix_funds)} 只")

# # 2. 获取包含成立日期的基金规模数据
# # 注意：需要分别获取不同类型，这里以股票型和混合型为例
# print("正在获取基金规模数据以补充成立日期...")
# all_funds_with_date = []

# for fund_type in ['股票型基金', '混合型基金']:
#     try:
#         # 获取指定类型的基金规模数据 [citation:10]
#         scale_df = ak.fund_scale_open_sina(symbol=fund_type)
#         # 只保留我们需要的列
#         scale_df = scale_df[['基金代码', '基金简称', '成立日期']]
#         all_funds_with_date.append(scale_df)
#         print(f"已获取 {fund_type} 数据，共 {len(scale_df)} 条")
#     except Exception as e:
#         print(f"获取 {fund_type} 数据失败: {e}")

# # 合并所有类型的规模数据
# if all_funds_with_date:
#     fund_scale = pd.concat(all_funds_with_date, ignore_index=True)
#     # 清洗：将成立日期转为日期格式，无法转换的设为 NaT
#     fund_scale['成立日期'] = pd.to_datetime(fund_scale['成立日期'], errors='coerce')
#     # 删除成立日期为空的行
#     fund_scale = fund_scale.dropna(subset=['成立日期'])
#     print(f"成功获取包含成立日期的基金数据: {len(fund_scale)} 条")
# else:
#     raise Exception("未能获取到任何基金的规模数据")

# # 3. 合并数据，筛选成立10年及以上的基金
# # 计算10年前的日期
# ten_years_ago = datetime.now() - timedelta(days=365*10)

# # 将两个DataFrame按基金代码进行内连接
# merged = pd.merge(stock_mix_funds, fund_scale, on='基金代码', how='inner')

# # 筛选成立10年以上的基金
# result = merged[merged['成立日期'] <= ten_years_ago]

# # 4. 输出结果
# print(f"\n找到符合条件的主动股票/混合型基金: {len(result)} 只")
# # 只保留关键的列，并去除可能的重复项
# final_result = result[['基金代码', '基金简称_x', '基金类型', '成立日期']].drop_duplicates()
# print(final_result.head(10))   # 打印前10条查看

# # 可选：保存到CSV文件
# # final_result.to_csv('qualified_funds.csv', index=False, encoding='utf-8-sig')

正在获取全量基金列表，请稍候...
筛选出股票/混合型基金: 1098 只
正在获取基金规模数据以补充成立日期...
已获取 股票型基金 数据，共 6431 条
已获取 混合型基金 数据，共 10000 条
成功获取包含成立日期的基金数据: 16431 条

找到符合条件的主动股票/混合型基金: 138 只
     基金代码       基金简称_x 基金类型       成立日期
0  000082   嘉实研究阿尔法股票A  股票型 2013-05-28
1  000309  大摩品质生活精选股票A  股票型 2013-10-29
2  000326   南方中小盘成长股票A  股票型 2015-10-28
3  000409     鹏华环保产业股票  股票型 2014-03-07
4  000411  景顺长城优质成长股票A  股票型 2014-01-02
5  000418  景顺长城成长之星股票A  股票型 2013-12-13
6  000457    摩根核心成长股票A  股票型 2014-02-10
7  000471     富国城镇发展股票  股票型 2014-01-28
8  000513  富国高端制造行业股票A  股票型 2014-06-20
9  000524    摩根民生需求股票A  股票型 2014-03-14


In [ ]:
# final_result

,基金代码,基金简称_x,基金类型,成立日期
0,000082,嘉实研究阿尔法股票A,股票型,2013-05-28
1,000309,大摩品质生活精选股票A,股票型,2013-10-29
2,000326,南方中小盘成长股票A,股票型,2015-10-28
3,000409,鹏华环保产业股票,股票型,2014-03-07
4,000411,景顺长城优质成长股票A,股票型,2014-01-02
...,...,...,...,...
147,002229,华夏经济转型股票,股票型,2016-03-15
149,002300,长盛医疗量化股票A,股票型,2016-02-03
152,002334,汇丰晋信大盘波动股票A,股票型,2016-03-11
153,002335,汇丰晋信大盘波动股票C,股票型,2016-03-11


In [ ]:
# import akshare as ak
# import pandas as pd
# from datetime import datetime, timedelta

# print("=" * 50)
# print("开始获取全量基金数据，请稍候...")
# print("=" * 50)

# # ------------------------------
# # 1. 获取包含状态的基础信息（使用旧接口，含基金状态）
# # ------------------------------
# try:
#     # 旧接口 fund_em_fund_name 包含“基金状态”列，数据全面
#     print("正在获取基金基础信息（含状态）...")
#     fund_basic_all = ak.fund_em_fund_name()
#     print(f"成功获取 {len(fund_basic_all)} 条基金基础信息")
# except Exception as e:
#     print("警告：无法使用 fund_em_fund_name 接口，尝试替代方案...")
#     # 替代方案：使用新版接口，但新版无状态，需额外补充（效率较低）
#     fund_basic_all = ak.fund_name_em()
#     # 为所有基金添加默认状态“正常”（后续需要进一步筛选，此处不完美但尽力）
#     fund_basic_all['基金状态'] = '正常'
#     print("使用替代接口，请注意状态字段可能不准确")

# # 确保列名统一（旧接口列名可能为中文）
# required_cols = ['基金代码', '基金简称', '基金类型', '基金状态']
# for col in required_cols:
#     if col not in fund_basic_all.columns:
#         print(f"错误：基础数据缺少必要列 {col}，请检查接口返回")
#         exit()

# # ------------------------------
# # 2. 筛选条件：
# #    - 基金类型为股票型或混合型
# #    - 基金状态为“正常”
# #    - 排除场内基金（类型包含 ETF、LOF、分级、封闭）
# # ------------------------------
# print("\n正在按条件筛选...")

# # 先筛选主动股票型（股票型、混合型）
# stock_mix = fund_basic_all[fund_basic_all['基金类型'].isin(['股票型', '混合型'])].copy()

# # 再筛选基金状态为“正常”
# stock_mix = stock_mix[stock_mix['基金状态'] == '正常']

# # 排除场内基金：类型中不能包含以下关键字（不区分大小写）
# exclude_keywords = ['ETF', 'LOF', '分级', '封闭']
# pattern = '|'.join(exclude_keywords)
# # 注意：某些基金类型可能含多个词，如“股票ETF”
# stock_mix = stock_mix[~stock_mix['基金类型'].str.contains(pattern, case=False, na=False)]

# print(f"初步筛选后剩余基金数: {len(stock_mix)} 只")

# # ------------------------------
# # 3. 获取成立日期数据（仍使用新浪规模接口）
# # ------------------------------
# print("\n正在获取基金成立日期信息...")
# all_funds_with_date = []

# for fund_type in ['股票型基金', '混合型基金']:
#     try:
#         scale_df = ak.fund_scale_open_sina(symbol=fund_type)
#         scale_df = scale_df[['基金代码', '基金简称', '成立日期']]
#         all_funds_with_date.append(scale_df)
#         print(f"已获取 {fund_type} 数据，共 {len(scale_df)} 条")
#     except Exception as e:
#         print(f"获取 {fund_type} 数据失败: {e}")

# if not all_funds_with_date:
#     raise Exception("未能获取任何成立日期数据，程序终止")

# fund_scale = pd.concat(all_funds_with_date, ignore_index=True)
# fund_scale['成立日期'] = pd.to_datetime(fund_scale['成立日期'], errors='coerce')
# fund_scale = fund_scale.dropna(subset=['成立日期'])

# # ------------------------------
# # 4. 合并并筛选成立十年以上
# # ------------------------------
# ten_years_ago = datetime.now() - timedelta(days=365*10)

# merged = pd.merge(stock_mix, fund_scale, on='基金代码', how='inner')
# result = merged[merged['成立日期'] <= ten_years_ago]

# print(f"\n最终符合条件的基金: {len(result)} 只")

# # 整理输出列（去除重复列名）
# result_final = result[['基金代码', '基金简称_x', '基金类型', '基金状态', '成立日期']].drop_duplicates()
# result_final = result_final.rename(columns={'基金简称_x': '基金简称'})

# # 按成立日期排序
# result_final = result_final.sort_values('成立日期')

# print("\n前10条结果展示:")
# print(result_final.head(10))

# # 可选：保存为CSV
# # result_final.to_csv('active_equity_10yr_offline_normal.csv', index=False, encoding='utf-8-sig')
# print("\n筛选完成！")

开始获取全量基金数据，请稍候...
正在获取基金基础信息（含状态）...
警告：无法使用 fund_em_fund_name 接口，尝试替代方案...
使用替代接口，请注意状态字段可能不准确

正在按条件筛选...
初步筛选后剩余基金数: 1098 只

正在获取基金成立日期信息...
已获取 股票型基金 数据，共 6431 条
已获取 混合型基金 数据，共 10000 条

最终符合条件的基金: 138 只

前10条结果展示:
      基金代码         基金简称 基金类型 基金状态       成立日期
0   000082   嘉实研究阿尔法股票A  股票型   正常 2013-05-28
1   000309  大摩品质生活精选股票A  股票型   正常 2013-10-29
5   000418  景顺长城成长之星股票A  股票型   正常 2013-12-13
4   000411  景顺长城优质成长股票A  股票型   正常 2014-01-02
7   000471     富国城镇发展股票  股票型   正常 2014-01-28
6   000457    摩根核心成长股票A  股票型   正常 2014-02-10
3   000409     鹏华环保产业股票  股票型   正常 2014-03-07
9   000524    摩根民生需求股票A  股票型   正常 2014-03-14
10  000549   华安大国新经济股票A  股票型   正常 2014-04-14
11  000577    安信价值精选股票A  股票型   正常 2014-04-21

筛选完成！


In [ ]:
# result_final = result_final.rename(columns={'name': '基金简称', 'ts_code': '基金代码', 'fund_type': '基金类型', 'found_date': '成立日期'})
# # result_final = result_final.drop(columns=['基金状态'], inplace=True)
# result_final

,基金代码,基金简称,基金类型,成立日期
0,000082,嘉实研究阿尔法股票A,股票型,2013-05-28
1,000309,大摩品质生活精选股票A,股票型,2013-10-29
5,000418,景顺长城成长之星股票A,股票型,2013-12-13
4,000411,景顺长城优质成长股票A,股票型,2014-01-02
7,000471,富国城镇发展股票,股票型,2014-01-28
...,...,...,...,...
152,002334,汇丰晋信大盘波动股票A,股票型,2016-03-11
139,001917,招商量化精选股票A,股票型,2016-03-15
143,001975,景顺长城环保优势股票,股票型,2016-03-15
147,002229,华夏经济转型股票,股票型,2016-03-15
